In [1]:
"""Train a layer-wise GRU exit classifier on summary predictions only.

Install: pip install torch transformers datasets sentencepiece
Run: python train_summary_exit_gru.py --samples 100 --epochs 3

One GPU (or CPU); frozen Llama, teacher-forced reference highlights.
This trains an exit classifier, not an actual early-exit decoding engine.
"""

import argparse
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

import itertools
import torch
import transformers
from datasets import load_dataset
from dataclasses import dataclass

model_path: str = "facebook/layerskip-llama2-7B"

# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# dtype = torch.float16 if device.type == "cuda" else torch.float32

# print(f"Using device: {device}")

tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_path,
    use_safetensors=True,
    device_map="auto",
    torch_dtype=torch.float16,
)

# model = transformers.AutoModelForCausalLM.from_pretrained(
#         model_path,
#         use_safetensors=True,
#         device_map="auto",
#         torch_dtype=(
#             torch.float16
#             if torch.cuda.is_available()
#             else torch.float32
#         ),
#         # Supports the explicit 4-D additive mask constructed above.
#         attn_implementation="eager",
#     )





/home/markphan24/anaconda3/envs/layer_skip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:10<00:00,  3.55s/it]


In [ ]:
def encode_example(tokenizer, prompt, target, task="summarize", max_length="4096"):
    """
    For summary task: Format: <s>[Prompt: ]  {prompt} [Summary: ] {highlight}</s>
    """
    encode = lambda text: tokenizer.encode(text, add_special_tokens=False)

    if tokenizer.bos_token_id is None or tokenizer.eos_token_id is None:
        raise ValueError("This example requires BOS and EOS tokens.")

    prefix = [tokenizer.bos_token_id] + encode("[Prompt: ]  ")
    if task == "summarize":
        separator = encode(" [Summary: ] ")
    target = encode(target) + [tokenizer.eos_token_id]

    # Truncate only the article; preserve the entire highlight and EOS.
    prompt_budget = max_length - len(prefix) - len(separator) - len(target)
    if prompt_budget < 1:
        raise ValueError("Highlight is too long; increase --max-length.")

    prompt_ids = prefix + encode(prompt)[:prompt_budget] + separator
    input_ids = prompt_ids + target

    # Shift manually: each position predicts the next token.
    # Only summary tokens and EOS contribute to the classifier's training.
    labels = [-100] * len(prompt_ids) + target
    labels = labels[1:] + [-100]

    return {"input_ids": input_ids, "labels": labels}


In [ ]:
def collate(examples, pad_token_id):  
    """Left padding. Attention and loss masks have different purposes."""
    length = max(len(x["input_ids"]) for x in examples)
    ids = torch.full((len(examples), length), pad_token_id, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    attention = torch.zeros_like(ids)
    for i, example in enumerate(examples):
        n = len(example["input_ids"])
        print(n)
        ids[i, -n:] = torch.tensor(example["input_ids"])
        labels[i, -n:] = torch.tensor(example["labels"])
        attention[i, -n:] = 1
    return {"input_ids": ids, "attention_mask": attention, "labels": labels}


In [ ]:
import torch


def encode_and_collate(
    tokenizer,
    prompt,
    target,
    task="summarize",
    max_length=4096,
):
    """Encode prompt/target pairs, shift labels, and left-pad the batch."""
    if tokenizer.bos_token_id is None or tokenizer.eos_token_id is None:
        raise ValueError("This function requires BOS and EOS tokens.")

    if task != "summarize":
        raise ValueError(f"Unsupported task: {task}")

    if not isinstance(max_length, int) or max_length < 1:
        raise ValueError("max_length must be a positive integer.")

    # Convert single strings into a batch of size 1.
    prompts = [prompt] if isinstance(prompt, str) else list(prompt)
    targets = [target] if isinstance(target, str) else list(target)

    if not prompts:
        raise ValueError("The batch must not be empty.")

    if len(prompts) != len(targets):
        raise ValueError("prompt and target must have the same batch size.")

    encode = lambda text: tokenizer.encode(text, add_special_tokens=False)

    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id

    prefix = [tokenizer.bos_token_id] + encode("[Prompt: ]  ")
    if task == "summarize":
        separator = encode(" [Summary: ] ")

    encoded_examples = []

    # 1. Encode each prompt/target pair.
    for prompt_text, target_text in zip(prompts, targets):
        target_ids = encode(target_text) + [tokenizer.eos_token_id]

        # Preserve the entire target and EOS; truncate only the prompt.
        prompt_budget = (
            max_length - len(prefix) - len(separator) - len(target_ids)
        )
        if prompt_budget < 1:
            raise ValueError("Target is too long; increase max_length.")

        prompt_ids = (
            prefix
            + encode(prompt_text)[:prompt_budget]
            + separator
        )
        input_ids = prompt_ids + target_ids

        # Supervise only target predictions, including EOS.
        # Each position's label is the next token.
        labels = [-100] * len(prompt_ids) + target_ids
        labels = labels[1:] + [-100]

        encoded_examples.append((input_ids, labels))

    # 2. Left-pad to the longest sequence in this batch.
    batch_size = len(encoded_examples)
    batch_length = max(len(ids) for ids, _ in encoded_examples)

    input_ids = torch.full(
        (batch_size, batch_length),
        pad_token_id,
        dtype=torch.long,
    )
    labels = torch.full_like(input_ids, -100)
    attention_mask = torch.zeros_like(input_ids)

    for i, (example_ids, example_labels) in enumerate(encoded_examples):
        n = len(example_ids)

        input_ids[i, -n:] = torch.tensor(example_ids, dtype=torch.long)
        labels[i, -n:] = torch.tensor(example_labels, dtype=torch.long)
        attention_mask[i, -n:] = 1

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [6]:
batch = encode_and_collate(
    tokenizer,
    prompt="Alice studies computer science at university.",
    target="Alice is a computer science student.",
)
batch

{'input_ids': tensor([[    1,   518, 18571,   415, 29901,  4514,   259, 16308, 11898,  6601,
          10466,   472, 16372, 29889, 29871,   518, 26289, 29901,  4514, 29871,
          16308,   338,   263,  6601, 10466,  8368, 29889,     2]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1]]),
 'labels': tensor([[ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
           -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100, 16308,
            338,   263,  6601, 10466,  8368, 29889,     2,  -100]])}

In [7]:
batch = encode_and_collate(
    tokenizer,
    prompt=[
        "Alice studies computer science at university.",
        "Heavy rain caused flooding across the city yesterday.",
    ],
    target=[
        "Alice is a computer science student.",
        "Heavy rain flooded the city.",
    ],
)
batch

{'input_ids': tensor([[    2,     2,     2,     2,     2,     1,   518, 18571,   415, 29901,
           4514,   259, 16308, 11898,  6601, 10466,   472, 16372, 29889, 29871,
            518, 26289, 29901,  4514, 29871, 16308,   338,   263,  6601, 10466,
           8368, 29889,     2],
         [    1,   518, 18571,   415, 29901,  4514,   259,   940,  5301, 17251,
           8581,  5685,  3689,  4822,   278,  4272, 22600, 29889, 29871,   518,
          26289, 29901,  4514, 29871,   940,  5301, 17251,  5685,  6797,   278,
           4272, 29889,     2]]),
 'attention_mask': tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1]]),
 'labels': tensor([[ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
           -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
           

In [ ]:
# example usage
examples = [
    encode_example(tokenizer, "The capital of France is", "Paris", max_length=4096),
    encode_example(tokenizer, "YouTube is created in what year", "2005", max_length=4096),
]
batch = collate(examples, pad_token_id=0)
batch

In [18]:

import itertools
from torch import version
import transformers
from datasets import load_dataset
from dataclasses import dataclass


DATASET = "abisee/cnn_dailymail"
VERSIONS = {"v1": "1.0.0", "v2": "2.0.0", "v3": "3.0.0"}
SPLITS = {"train": "train", "val": "validation", "test": "test"}

# 1. Load the dataset - create shards
# Each row in a shard contains input_ids, input_text, 
def load_records(args):
    """Stream records from a zero-based row index in the selected split."""
    if args.start_row < 0:
        raise ValueError("start_row must be non-negative")
    dataset = load_dataset(
        args.dataset,
        name=VERSIONS[args.version],
        split=SPLITS[args.split],
        streaming=True,
    )
    if args.start_row > 0:
        dataset = dataset.skip(args.start_row)
    records = iter(dataset)
    if args.max_documents is not None:
        records = itertools.islice(records, args.max_documents)
    return records


# temporary
@dataclass
class Argument:
    dataset: str
    max_documents: int
    start_row: int
    version: str = "v1"
    split: str = "test"

args = Argument(dataset=DATASET, max_documents=3, start_row=0, version="v1", split="test")

for record in load_records(args):
    print(record)


{'article': '(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednesday

In [ ]:
# 1. Load the dataset - create shards
# Each row in a shard contains input_ids, input_text, 
def load_records(args):
    """Stream records from a zero-based row index in the selected split."""
    if args.start_row < 0:
        raise ValueError("start_row must be non-negative")
    dataset = load_dataset(
        args.dataset,
        name=VERSIONS[args.version],
        split=SPLITS[args.split],
        streaming=True,
    )
    if args.start_row > 0:
        dataset = dataset.skip(args.start_row)
    records = iter(dataset)
    if args.max_documents is not None:
        records = itertools.islice(records, args.max_documents)
    return records


# temporary
@dataclass
class Argument:
    dataset: str
    max_documents: int
    start_row: int
    version: str = "v1"
    split: str = "test"

args = Argument(dataset=DATASET, max_documents=1000, start_row=0)

records = []
for record in load_records(args):
    records.append(record)

In [ ]:
# Model - still need revision

class EarlyExitGRU(nn.Module):
    def __init__(self, llm_hidden_size=4096, gru_hidden_size=128):
        super().__init__()
        self.input_norm = nn.LayerNorm(llm_hidden_size)
        # The sequence dimension is decoder depth, NOT text positions.
        # Unidirectional: decision at layer l sees only layers 1,...,l.
        self.gru = nn.GRU(llm_hidden_size, gru_hidden_size, batch_first=True)
        self.exit_head = nn.Linear(gru_hidden_size, 1)

    def forward(self, hidden_states, recurrent_state=None):
        # hidden_states: [number_of_target_tokens, early_layers, hidden_size]
        outputs, state = self.gru(self.input_norm(hidden_states), recurrent_state)
        return self.exit_head(outputs).squeeze(-1), state

    @torch.no_grad()
    def step(self, layer_hidden, recurrent_state=None):
        """One layer of one decoding step; reset state for each new token."""
        logits, state = self(layer_hidden.unsqueeze(1).float(), recurrent_state)
        return logits[:, 0].sigmoid(), state


In [ ]:

@torch.no_grad()
def extract_summary_features(model, batch, projection_chunk_size=32):
    """Return CPU features [N,L-1,H] and binary exit labels [N,L-1].

    For standard Hugging Face Llama:
      hidden_states[0] = embeddings
      hidden_states[1:L] = raw outputs of layers 1,...,L-1
      last_hidden_state = final layer output AFTER final RMSNorm
    Thus final states must NOT be normalized a second time.
    """
    model.eval()
    device = model.get_input_embeddings().weight.device
    ids = batch["input_ids"].to(device)
    attention = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # State at position t predicts token at position t+1.
    # This selects the BOS state for the first highlight token.
    target_mask = labels[:, 1:].ne(-100)
    target_mask &= attention[:, :-1].bool() & attention[:, 1:].bool()
    rows, positions = target_mask.nonzero(as_tuple=True)
    if rows.numel() == 0:
        raise ValueError("Batch contains no summary targets.")

    outputs = model.model(
        input_ids=ids,
        attention_mask=attention,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )
    num_layers = len(model.model.layers)
    if len(outputs.hidden_states) != num_layers + 1:
        raise ValueError("Unexpected Llama hidden-state layout.")

    def predict_tokens(states, normalize):
        # Project only selected summary-prediction positions, in chunks.
        predictions = []
        for start in range(0, rows.numel(), projection_chunk_size):
            stop = start + projection_chunk_size
            h = states[rows[start:stop], positions[start:stop]]
            if normalize:
                h = model.model.norm(h)
            predictions.append(model.lm_head(h).argmax(-1).cpu())
        return torch.cat(predictions)

    teacher_ids = predict_tokens(outputs.last_hidden_state, normalize=False)
    features, exit_targets = [], []
    for layer_index in range(1, num_layers):
        raw_hidden = outputs.hidden_states[layer_index]
        features.append(raw_hidden[rows, positions].cpu())
        early_ids = predict_tokens(raw_hidden, normalize=True)
        exit_targets.append(early_ids.eq(teacher_ids).float())

    # Llama stays frozen. No full [B,T,V] logits are stored for each layer.
    return torch.stack(features, dim=1), torch.stack(exit_targets, dim=1)


def train_epoch(model, classifier, loader, optimizer, token_batch_size=32):
    classifier.train()
    device = next(classifier.parameters()).device
    total_loss, count = 0.0, 0
    for batch in loader:
        features, targets = extract_summary_features(model, batch)
        order = torch.randperm(features.shape[0])
        for indices in order.split(token_batch_size):
            x = features[indices].to(device=device, dtype=torch.float32)
            y = targets[indices].to(device)
            optimizer.zero_grad(set_to_none=True)
            # GRU state resets here: each target token is an independent
            # sequence across Llama layers, with no cross-token recurrence.
            exit_logits, _ = classifier(x)
            loss = F.binary_cross_entropy_with_logits(exit_logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * y.numel()
            count += y.numel()
    return total_loss / max(count, 1)


def main():
    from functools import partial
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer

    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", default="facebook/layerskip-llama2-7B")
    parser.add_argument("--samples", type=int, default=100)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--token-batch-size", type=int, default=32)
    parser.add_argument("--max-length", type=int, default=1024)
    parser.add_argument("--gru-hidden-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--output", default="summary_exit_gru.pt")
    args = parser.parse_args()
    for name in ("samples", "epochs", "batch_size", "token_batch_size",
                 "max_length", "gru_hidden_size"):
        if getattr(args, name) <= 0:
            parser.error(f"{name} must be positive")
    torch.manual_seed(42)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if device.type == "cuda" else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(args.model)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    # Deliberately single-device: features, RMSNorm, and LM head must agree.
    model = AutoModelForCausalLM.from_pretrained(
        args.model, torch_dtype=dtype, attn_implementation="sdpa",
    ).to(device)
    model.requires_grad_(False)
    model.eval()
    if args.max_length > model.config.max_position_embeddings:
        parser.error("max-length exceeds the model context limit")
    data = load_dataset(
        "abisee/cnn_dailymail", "3.0.0", split=f"train[:{args.samples}]"
    )
    examples = [encode_example(tokenizer, row["article"], row["highlights"],
                               args.max_length) for row in data]
    loader = DataLoader(
        examples, batch_size=args.batch_size, shuffle=True,
        collate_fn=partial(collate, pad_token_id=tokenizer.pad_token_id),
    )
    classifier = EarlyExitGRU(model.config.hidden_size, args.gru_hidden_size).to(device)
    optimizer = torch.optim.AdamW(classifier.parameters(), lr=args.lr)
    for epoch in range(args.epochs):
        loss = train_epoch(model, classifier, loader, optimizer, args.token_batch_size)
        print(f"Epoch {epoch + 1}: summary-token exit BCE = {loss:.6f}", flush=True)
    destination = Path(args.output)
    destination.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "state_dict": {k: v.detach().cpu() for k, v in classifier.state_dict().items()},
        "llm_hidden_size": model.config.hidden_size,
        "gru_hidden_size": args.gru_hidden_size,
        "num_early_layers": len(model.model.layers) - 1,
        "base_model": args.model,
        "label_definition": "intermediate argmax equals final-layer argmax",
        "training_args": vars(args),
    }, destination)
    print(f"Saved {destination}")


if __name__ == "__main__":
    main()


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
eos = model.config.eos_token_id

# Illustrative token IDs. In your dataset, use your tokenizer's IDs.
# Both examples are padded to length 6 so they can be stacked.
dataset = [
    {
        #             prompt       summary   EOS
        "inputs_ids":      [10, 11, 12,  20, 21, eos],
        "attention_masked": [1,  1,  1,   1,  1,   1],
        "label":         [-100, -100, -100, 20, 21, eos],
    },
    {
        #             prompt  summary  EOS  padding
        "inputs_ids":      [30, 31,  40, eos, eos, eos],
        "attention_masked": [1,  1,   1,   1,   0,   0],
        "label":         [-100, -100, 40, eos, -100, -100],
    },
    # write 3 more data
    {
        "inputs_ids":      [50, 51,  60, eos, eos, eos],
        "attention_masked": [1,  1,   1,   1,   0,   0],
        "label":         [-100, -100, 60, eos, -100, -100],
    },
    {
        "inputs_ids":      [70, 71,  80, eos, eos, eos],
        "attention_masked": [1,  1,   1,   1,   0,   0],
        "label":         [-100, -100, 80, eos, -100, -100],
    },
    {
        "inputs_ids":      [90, 91, 100, eos, eos, eos],
        "attention_masked": [1,  1,   1,   1,   0,   0],
        "label":         [-100, -100, 100, eos, -100, -100],
    }

]

def collate(rows):
    # Map your dictionary keys to the argument names the model expects.
    return {
        "input_ids": torch.tensor([row["inputs_ids"] for row in rows]),
        "attention_mask": torch.tensor([row["attention_masked"] for row in rows]),
        "labels": torch.tensor([row["label"] for row in rows]),
    }

loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()
# for epoch in range(3):
for batch in loader:
    batch = {key: value.to(device) for key, value in batch.items()}
    print(batch)
    optimizer.zero_grad()
    loss = model(**batch).loss
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}, loss = {loss.item():.4f}")

{'input_ids': tensor([[   10,    11,    12,    20,    21, 50256],
        [   50,    51,    60, 50256, 50256, 50256]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0]], device='cuda:0'), 'labels': tensor([[ -100,  -100,  -100,    20,    21, 50256],
        [ -100,  -100,    60, 50256,  -100,  -100]], device='cuda:0')}


In [10]:
"""Stream CNN/DailyMail with manually shifted labels and dynamic left padding.

Install: pip install torch transformers datasets accelerate sentencepiece
Run: python efficient_cnn_loader.py

The example runs teacher-forced inference, not training or generation.
No dataset/model downloads happen when this module is imported.
"""

from dataclasses import dataclass

import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


DATASET = "abisee/cnn_dailymail"
VERSIONS = {"v1": "1.0.0", "v2": "2.0.0", "v3": "3.0.0"}
SPLITS = {"train": "train", "val": "validation", "test": "test"}


@dataclass
class Argument:
    dataset: str = DATASET
    max_documents: int | None = 3
    start_row: int = 0
    version: str = "v1"
    split: str = "test"
    batch_size: int = 2
    max_length: int = 4096
    model_path: str = "facebook/layerskip-llama2-7B"


def load_records(args):
    """Return a reusable streaming dataset, starting at a zero-based row."""
    if args.start_row < 0:
        raise ValueError("start_row must be non-negative.")
    if args.max_documents is not None and args.max_documents < 0:
        raise ValueError("max_documents must be non-negative or None.")

    dataset = load_dataset(
        args.dataset,
        name=VERSIONS[args.version],
        split=SPLITS[args.split],
        streaming=True,
    )
    if args.start_row:
        dataset = dataset.skip(args.start_row)
    if args.max_documents is not None:
        dataset = dataset.take(args.max_documents)
    return dataset


class TokenEncoder:
    """Cache fixed tokens and batch-tokenize variable text once per batch."""

    def __init__(self, tokenizer, task="summarize", max_length=4096):
        if tokenizer.bos_token_id is None or tokenizer.eos_token_id is None:
            raise ValueError("This function requires BOS and EOS tokens.")
        if task != "summarize":
            raise ValueError(f"Unsupported task: {task}")
        if not isinstance(max_length, int) or max_length < 1:
            raise ValueError("max_length must be a positive integer.")

        self.tokenizer = tokenizer
        self.max_length = max_length
        self.eos_id = tokenizer.eos_token_id
        self.pad_id = tokenizer.pad_token_id
        if self.pad_id is None:
            self.pad_id = self.eos_id

        self.prefix = [tokenizer.bos_token_id] + tokenizer.encode(
            "[Prompt: ]  ", add_special_tokens=False
        )
        self.separator = tokenizer.encode(
            " [Summary: ] ", add_special_tokens=False
        )

    def encode(self, prompt, target):
        prompts = [prompt] if isinstance(prompt, str) else list(prompt)
        targets = [target] if isinstance(target, str) else list(target)
        if not prompts:
            raise ValueError("The batch must not be empty.")
        if len(prompts) != len(targets):
            raise ValueError("prompt and target must have the same batch size.")

        batch_size = len(prompts)
        # Keep the pieces separate to preserve your original token boundaries.
        token_lists = self.tokenizer(
            prompts + targets,
            add_special_tokens=False,
            padding=False,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
        )["input_ids"]

        examples = []
        for i in range(batch_size):
            target_ids = token_lists[batch_size + i] + [self.eos_id]
            prompt_budget = (
                self.max_length - len(self.prefix)
                - len(self.separator) - len(target_ids)
            )
            if prompt_budget < 1:
                raise ValueError("Target is too long; increase max_length.")

            prompt_ids = (
                self.prefix + token_lists[i][:prompt_budget] + self.separator
            )
            examples.append((prompt_ids + target_ids, len(target_ids)))

        batch_length = max(len(ids) for ids, _ in examples)
        input_ids = torch.full(
            (batch_size, batch_length), self.pad_id, dtype=torch.long
        )
        attention_mask = torch.zeros_like(input_ids)
        labels = torch.full_like(input_ids, -100)

        for i, (ids, target_length) in enumerate(examples):
            start = batch_length - len(ids)
            input_ids[i, start:] = torch.tensor(ids, dtype=torch.long)
            attention_mask[i, start:] = 1

            # All sequences end with target + EOS. Copy those IDs one position
            # left: the last separator token predicts the first target token.
            target_start = batch_length - target_length
            labels[i, target_start - 1 : -1] = input_ids[i, target_start:]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

    def __call__(self, examples):
        return self.encode(
            [example["article"] for example in examples],
            [example["highlights"] for example in examples],
        )


def encode_and_collate(tokenizer, prompt, target, task="summarize", max_length=4096):
    """Compatibility wrapper; reuse SummaryCollator for repeated calls."""
    return TokenEncoder(tokenizer, task, max_length).encode(prompt, target)


def make_dataloader(tokenizer, args):
    return DataLoader(
        load_records(args),
        batch_size=args.batch_size,
        collate_fn=TokenEncoder(tokenizer, max_length=args.max_length),
        num_workers=0,  # Preserve exact stream order and skip/take semantics.
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )


# def main():
#     args = Argument()
#     tokenizer = AutoTokenizer.from_pretrained(args.model_path)
#     loader = make_dataloader(tokenizer, args)
#     model = AutoModelForCausalLM.from_pretrained(
#         args.model_path,
#         use_safetensors=True,
#         device_map="auto",
#         torch_dtype=torch.float16,
#     ).eval()
#     device = model.get_input_embeddings().weight.device

#     with torch.inference_mode():
#         for batch in loader:
#             input_ids = batch["input_ids"].to(device, non_blocking=True)
#             mask = batch["attention_mask"].to(device, non_blocking=True)
#             # Real tokens start at position zero, regardless of left padding.
#             position_ids = (mask.cumsum(-1) - 1).masked_fill(mask == 0, 0)
#             outputs = model(
#                 input_ids=input_ids,
#                 attention_mask=mask,
#                 position_ids=position_ids,
#                 use_cache=False,
#             )
#             # Do NOT pass batch['labels']: the model would shift them again.
#             # For a custom loss, logits[:, t] pairs with labels[:, t] directly.
#             print("input_ids:", tuple(input_ids.shape))
#             print("logits:", tuple(outputs.logits.shape))
#             print("supervised tokens:", (batch["labels"] != -100).sum().item())
#             del outputs


# if __name__ == "__main__":
#     main()


In [ ]:
import argparse
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

import itertools
import torch
import transformers
from datasets import load_dataset
from dataclasses import dataclass

model_path: str = "facebook/layerskip-llama2-7B"

# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# dtype = torch.float16 if device.type == "cuda" else torch.float32

# print(f"Using device: {device}")

tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_path,
    use_safetensors=True,
    device_map="auto",
    torch_dtype=torch.float16,
)

In [11]:
args = Argument()
output = make_dataloader(tokenizer, args)

In [14]:
output

In [19]:
i = 1

# Remove padding using the attention mask.
real_tokens = batch["attention_mask"][i].bool()
input_ids = batch["input_ids"][i][real_tokens]

print("INPUT TEXT:")
print(tokenizer.decode(input_ids.tolist(), skip_special_tokens=False))

# Remove ignored positions before decoding labels.
labels = batch["labels"][i]
target_ids = labels[labels != -100]

print("\nTARGET TEXT:")
print(tokenizer.decode(target_ids.tolist(), skip_special_tokens=False))

print(f"{'Pos':>5}  {'Input token':<25}  Next-token label")

for pos in real_tokens.nonzero(as_tuple=True)[0].tolist():
    input_id = batch["input_ids"][i, pos].item()
    label_id = batch["labels"][i, pos].item()

    input_token = tokenizer.convert_ids_to_tokens(input_id)
    label_token = (
        "IGNORED"
        if label_id == -100
        else tokenizer.convert_ids_to_tokens(label_id)
    )

    print(f"{pos:>5}  {input_token!r:<25}  {label_token}")

INPUT TEXT:
<s> [Prompt: ]   (CNN)Never mind cats having nine lives. A stray pooch in Washington State has used up at least three of her own after being hit by a car, apparently whacked on the head with a hammer in a misguided mercy killing and then buried in a field -- only to survive. That's according to Washington State University, where the dog -- a friendly white-and-black bully breed mix now named Theia -- has been receiving care at the Veterinary Teaching Hospital. Four days after her apparent death, the dog managed to stagger to a nearby farm, dirt-covered and emaciated, where she was found by a worker who took her to a vet for help. She was taken in by Moses Lake, Washington, resident Sara Mellado. "Considering everything that she's been through, she's incredibly gentle and loving," Mellado said, according to WSU News. "She's a true miracle dog and she deserves a good life." Theia is only one year old but the dog's brush with death did not leave her unscathed. She suffered a d

In [22]:
import math

scores = [2.0, 1.0, -6500]

# Step 1: Exponentiate each score.
exps = [math.exp(x) for x in scores]
print("Exponentials:", exps)  # [7.389..., 2.718..., 0.0]

# Step 2: Sum the exponentials.
total = sum(exps)
print("Sum:", total)  # 10.107...

# Step 3: Divide each exponential by the sum.
probabilities = [x / total for x in exps]
print("Softmax:", probabilities)  # [0.731..., 0.268..., 0.0]

Exponentials: [7.38905609893065, 2.718281828459045, 0.0]
Sum: 10.107337927389695
Softmax: [0.7310585786300049, 0.2689414213699951, 0.0]


In [17]:
import torch
from torch import nn

class EarlyExitGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True, bidirectional=False,
            num_layers=10,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        states, hn = self.gru(x)
        return self.head(hn[-1]).squeeze(-1)  # Use the last layer's hidden state for classification

In [27]:
model = EarlyExitGRU(4096, 128)
# Count total parameters
total_params = sum(p.numel() for p in model.parameters())

# Count ONLY trainable parameters (weights that update during training)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params}")
print(f"Trainable Parameters: {trainable_params}")

Total Parameters: 2514561
Trainable Parameters: 2514561


In [31]:
import pandas as pd

df = pd.read_parquet("cnn_layer_outputs.parquet")
df

,input_ids,labels,attention_mask,reference_target_ids,hidden_outputs,prediction_ids,matches_reference_or_final,distributions
0,"[1, 518, 18571, 415, 29962, 29871, 313, 29907,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[341, 1590, 10475, 4076, 278, 306, 4174, 24894...","[[[0.00019836426, 0.0044174194, -0.012626648, ...","[[29906, 29889, 10475, 29892, 278, 29871, 626,...","[[False, False, True, False, True, False, Fals...","[[[2.028762e-10, 8.2463397e-10, 1.9790803e-06,..."
1,"[1, 518, 18571, 415, 29962, 29871, 313, 29907,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[450, 423, 29892, 263, 8227, 368, 2078, 287, 6...","[[[0.00198555, 0.003627777, -0.012908936, -0.0...","[[29896, 29871, 471, 29871, 29871, 29890, 2989...","[[False, False, False, False, False, False, Fa...","[[[2.7030617e-10, 1.2547751e-09, 1.4929351e-06..."
2,"[1, 518, 18571, 415, 29962, 29871, 313, 29907,...","[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[12929, 4850, 328, 435, 485, 328, 22453, 361, ...","[[[-0.0002193451, 0.0043640137, -0.009841919, ...","[[29906, 28913, 328, 10475, 29889, 713, 29892,...","[[False, False, True, False, False, False, Fal...","[[[1.8940291e-10, 7.943073e-10, 1.4684047e-06,..."


In [34]:
# write df that only takes hidden_outputs and matches_reference_or_final, and discard other columns
df_new = df[['hidden_outputs', 'matches_reference_or_final']]
df_new.to_parquet("cnn_layer_outputs_filtered.parquet", index=False)

In [ ]:
import pandas as pd
df = pd.read_parquet("cnn_layer_outputs_filtered.parquet")
df["matches_reference_or_final"].shape  # == 3? Why?

,hidden_outputs,matches_reference_or_final
0,"[[[0.00019836426, 0.0044174194, -0.012626648, ...","[[False, False, True, False, True, False, Fals..."
1,"[[[0.00198555, 0.003627777, -0.012908936, -0.0...","[[False, False, False, False, False, False, Fa..."
2,"[[[-0.0002193451, 0.0043640137, -0.009841919, ...","[[False, False, True, False, False, False, Fal..."


In [14]:
df["hidden_outputs"][0][1].shape

(52,)

In [15]:
df["hidden_outputs"][1][1].shape

(66,)

In [16]:
df["hidden_outputs"][2][1].shape

(52,)

In [ ]:
model = EarlyExitGRU(4096, 128)
# write code that takes in a batch of hidden_outputs and matches_reference_or_final, and returns the loss for the batch using BCEWithLogitsLoss
x = torch.stack(torch.tensor(df["hidden_outputs"].tolist()[0]))  # shape: [batch_size, num_layers, hidden_size]
y = torch.stack(torch.tensor(df["matches_reference_or_final"].tolist()[0]))  # shape: [batch_size, num_layers]

loss = nn.BCEWithLogitsLoss()(model(x), y)

print(loss)


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [65]:
import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

df = pd.read_parquet("cnn_layer_outputs_filtered.parquet")


class EarlyExitGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=10):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
        )
        self.head = nn.Linear(hidden_size, 1)

    # def forward(self, x):
    #     # x: [B, L, T, H]
    #     B, L, T, H = x.shape

    #     # Each token becomes an independent sequence across LLM layers.
    #     x = x.permute(0, 2, 1, 3).reshape(B * T, L, H)

    #     states, _ = self.gru(x)                  # [B*T, L, hidden_size]
    #     logits = self.head(states).squeeze(-1)  # [B*T, L]

    #     return logits.reshape(B, T, L).transpose(1, 2)  # [B, L, T]

    def forward(self, x):
        # x: [B, L, T, H]
        B, L, T, H = x.shape

        # Each token becomes an independent sequence across LLM layers.
        x = x.permute(0, 2, 1, 3).reshape(B * T, L, H)

        states, last_hidden = self.gru(x)                  # [B*T, L, hidden_size]
        logits = self.head(last_hidden)

        return logits.reshape(B, T, L).transpose(1, 2)  # [B, L, T]


def to_tensor(value):
    """Handle nested lists and NumPy object arrays loaded from Parquet."""
    if isinstance(value, np.ndarray) and value.dtype != object:
        return torch.tensor(value, dtype=torch.float32)
    if isinstance(value, (list, tuple, np.ndarray)):
        return torch.stack([to_tensor(item) for item in value])
    return torch.tensor(value, dtype=torch.float32)


def collate_batch(indices):
    xs, ys = [], []

    for index in indices:
        row = df.iloc[index]

        x = to_tensor(row["hidden_outputs"])             # [L, T, H]
        y = to_tensor(row["matches_reference_or_final"]) # [L, T]

        assert x.ndim == 3 and y.ndim == 2
        assert x.shape[:2] == y.shape

        # pad_sequence expects the variable-length dimension first.
        xs.append(x.transpose(0, 1))  # [T, L, H]
        ys.append(y.transpose(0, 1))  # [T, L]

    X = pad_sequence(xs, batch_first=True, padding_value=0)
    y = pad_sequence(ys, batch_first=True, padding_value=-100)

    return X.transpose(1, 2), y.transpose(1, 2)


loader = DataLoader(
    range(len(df)),
    batch_size=3,
    shuffle=True,
    collate_fn=collate_batch,
)

# Get one batch.
X, y = next(iter(loader))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize once, before training.
model = EarlyExitGRU(input_size=4096, hidden_size=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

model.train()

for epoch in range(3):
    for X, y in loader:
        X = X.to(device)
        y = y.to(device)

        valid_mask = y != -100

        if not valid_mask.any():
            continue

        optimizer.zero_grad(set_to_none=True)

        logits = model(X)  # [B, L, T], matching y

        # Only real token positions contribute to the loss.
        loss = criterion(logits[valid_mask], y[valid_mask])

        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch + 1} | Loss: {loss.item():.4f}")

RuntimeError: shape '[3, 66, 32]' is invalid for input of size 1980

In [37]:
X[0][1]
# check until what index first appear are zero vector
torch.nonzero(X[0][1] == 0, as_tuple=False)

tensor([[   4, 1819],
        [   7, 4056],
        [  11, 2541],
        ...,
        [  65, 4093],
        [  65, 4094],
        [  65, 4095]], device='cuda:0')

In [54]:
result = nn.BCEWithLogitsLoss(X[0][1][65] @ torch.rand(4096).to(device), y[0][1][65])# random vector to check if it is zero vector
result

BCEWithLogitsLoss()

In [50]:
y[0][1][65]

tensor(-100., device='cuda:0')

In [55]:
X[0][1][65]

tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0')